# INF4058 Data Engineering - Lab 3 Assignment
## Clean the NYC Taxi data, then stop

**Student Name:** Kareem Alaa Mohamed  
**Student ID:** 692300273  
**Course:** INF4058 Data Engineering  
**Submission:** PDF Report + GitHub Repo Link  

---
### Assignment Objectives
1. **Look**: Inspect the first 100 rows of `yellow_tripdata_2021-01.csv.gz`.
2. **Profile**: Read the entire file in chunks of 100,000 rows without loading all into RAM; compute nulls, dtypes, min, and max.
3. **Find Problems**: Identify and quantify all anomalies (bad dates, zero/negative fares and distances, passenger counts, duplicates).
4. **Decide**: Formulate data governance decisions (drop, fix, or flag) with explicit business rationale and owners.
5. **Clean**: Implement non-destructive `clean(df)` returning `(cleaned_df, dict{rule: rows_removed})`.
6. **Stop Here**: Profile and clean in-memory / staging without loading into PostgreSQL.

## Step 1: Look
Read the first 100 rows of `yellow_tripdata_2021-01.csv.gz`.
- What is a row?
- Which columns are dates?
- Which look empty?

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Read first 100 rows
df_sample = pd.read_csv('yellow_tripdata_2021-01.csv.gz', nrows=100)

print('--- PRINTED: df.head() (First 5 Rows) ---')
print(df_sample.head())

print('\n--- PRINTED: df.dtypes ---')
print(df_sample.dtypes)


--- PRINTED: df.head() (First 5 Rows) ---
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  total_amount  congestion_surcharge
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5        0.00           0.0                    0.3         11.80                   2.5
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5        0.00           0.0                    0.3          4.30                   0.0
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132     

### Analysis of Step 1
1. **What is a row?**  
   A row represents a single commercial taxi trip record completed by an authorized NYC Yellow Taxi vehicle, capturing pickup/dropoff timestamps, meter-computed distances, spatial zone coordinates (`PULocationID`, `DOLocationID`), rate classification (`RatecodeID`), payment method (`payment_type`), and itemized monetary breakdowns (`fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, and `congestion_surcharge`).

2. **Which columns are dates?**  
   - `tpep_pickup_datetime`: Date and time when the passenger was picked up.
   - `tpep_dropoff_datetime`: Date and time when the passenger was dropped off.

3. **Which look empty?**  
   In the first 100 rows, no columns are completely empty (0 nulls). However, `passenger_count` displays a value of `0` in row index 3, and later chunks (chunks 13-14) exhibit 98,352 null values in `VendorID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, and `payment_type`.

## Step 2: Profile
Read the whole file in chunks of 100,000 rows. Never load it all at once. Do not modify the data.

In [2]:
chunksize = 100000
col_stats = {}
total_rows = 0
parse_dates = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']

for chunk in pd.read_csv('yellow_tripdata_2021-01.csv.gz', chunksize=chunksize, parse_dates=parse_dates, low_memory=False):
    total_rows += len(chunk)
    for col in chunk.columns:
        if col not in col_stats:
            col_stats[col] = {
                'dtype': str(chunk[col].dtype),
                'nulls': 0,
                'min': None,
                'max': None
            }
        col_stats[col]['nulls'] += int(chunk[col].isnull().sum())
        valid = chunk[col].dropna()
        if len(valid) > 0:
            c_min = valid.min()
            c_max = valid.max()
            if col_stats[col]['min'] is None or c_min < col_stats[col]['min']:
                col_stats[col]['min'] = c_min
            if col_stats[col]['max'] is None or c_max > col_stats[col]['max']:
                col_stats[col]['max'] = c_max

profile_df = pd.DataFrame([
    {
        'column': col,
        'dtype': stats['dtype'],
        'null_count': stats['nulls'],
        'null_pct': round((stats['nulls'] / total_rows) * 100, 2),
        'min_val': str(stats['min']),
        'max_val': str(stats['max'])
    }
    for col, stats in col_stats.items()
])

print('=== PROFILING TABLE (All 1,369,765 rows profiled in chunks of 100,000) ===')
print(profile_df.to_string(index=False))


=== PROFILING TABLE (All 1,369,765 rows profiled in chunks of 100,000) ===
               column          dtype  null_count  null_pct             min_val             max_val
             VendorID          int64       98352      7.18                   1                   2
 tpep_pickup_datetime datetime64[us]           0      0.00 2008-12-31 23:05:14 2021-02-22 16:52:16
tpep_dropoff_datetime datetime64[us]           0      0.00 2008-12-31 23:07:22 2021-02-22 16:56:15
      passenger_count          int64       98352      7.18                   0                   8
        trip_distance        float64           0      0.00                 0.0           263163.28
           RatecodeID          int64       98352      7.18                   1                  99
   store_and_fwd_flag            str       98352      7.18                   N                   Y
         PULocationID          int64           0      0.00                   1                 265
         DOLocationID          int

## Step 3: Find Problems
Scan for pickups outside January 2021, negative/zero fares and distances, `passenger_count = 0`, and duplicate rows.

In [3]:
from clean_taxi_data import find_problems
problems_df = find_problems('yellow_tripdata_2021-01.csv.gz', chunksize=100000)


--- PRINTED TABLE: Data Quality Anomalies Identified Across Chunks ---
                                        Anomaly / Data Issue  Affected Rows  Percentage (%)
    Pickups outside Jan 2021 (< 2021-01-01 or >= 2021-02-01)             24          0.0018
       Dropoff datetime before pickup datetime (time travel)           5642          0.4119
                  Negative fare amount (fare_amount < $0.00)           6769          0.4942
                     Zero fare amount (fare_amount == $0.00)            642          0.0469
            Negative trip distance (trip_distance < 0 miles)              0          0.0000
               Zero trip distance (trip_distance == 0 miles)          19952          1.4566
             Passenger count equals 0 (passenger_count == 0)          26726          1.9511
                   Missing / Null passenger_count & VendorID          98352          7.1802
            RatecodeID == 99 (invalid / undefined rate code)             36          0.0026
         

## Step 4: Decisions Table
For each problem choose: **drop**, **fix** or **flag**. Say why, and who should own that rule.

In [4]:
decisions = [
    {
        'Problem': 'Pickups outside Jan 2021 (2008, Feb 2021)',
        'Decision': 'DROP',
        'Why': 'Preserves temporal partition boundary integrity. Downstream January analytics and billing aggregations will be corrupted by out-of-boundary rides.',
        'Owner': 'Data Engineering / Ingestion Pipeline Contract'
    },
    {
        'Problem': 'Dropoff before pickup (5,642 rows)',
        'Decision': 'DROP',
        'Why': 'Negative duration violates physical reality and causality. Distorts transit speed, route modeling, and driver turnaround times.',
        'Owner': 'Data Engineering / Quality Assurance'
    },
    {
        'Problem': 'Negative fares (6,769 rows)',
        'Decision': 'DROP',
        'Why': 'Represents chargebacks, disputes, meter cancellations, or system corrections. Pollutes transactional revenue and average trip yield metrics.',
        'Owner': 'Finance / Revenue & Ingestion Team'
    },
    {
        'Problem': 'Zero fares (642 rows)',
        'Decision': 'DROP',
        'Why': 'Non-revenue test rides, employee shuttles, or meter calibration runs. Non-commercial records bias passenger pricing models.',
        'Owner': 'Commercial Operations Team'
    },
    {
        'Problem': 'Zero trip distance (19,952 rows)',
        'Decision': 'DROP',
        'Why': 'Trips with zero distance traveled are stationary cancellations or GPS disconnects. Unusable for mobility, distance, or spatial analysis.',
        'Owner': 'Fleet Telematics / Operations Team'
    },
    {
        'Problem': 'Passenger count == 0 (26,726 rows)',
        'Decision': 'FIX',
        'Why': 'Taxi drivers frequently neglect to press the passenger count button on the meter. Imputing to 1 (statistical mode) salvages valid paid revenue trips.',
        'Owner': 'Data Analytics & ML Feature Engineering'
    },
    {
        'Problem': 'Missing values in VendorID & passenger_count (98,352 rows)',
        'Decision': 'DROP',
        'Why': 'Incomplete dispatch records lacking primary attribution and operational metadata. Violates relational completeness for target schema.',
        'Owner': 'Source Data Provider / Ingestion Team'
    },
    {
        'Problem': 'Duplicate rows (0 exact, 5,900 key dups)',
        'Decision': 'DROP',
        'Why': 'Guarantees pipeline idempotency against network retries, container restarts, or duplicate source file deliveries.',
        'Owner': 'Data Engineering Platform Team'
    }
]
decisions_df = pd.DataFrame(decisions)
print('--- PRINTED TABLE: Decisions Table (Problem, Decision, Why, Owner) ---')
print(decisions_df.to_string(index=False))


--- PRINTED TABLE: Decisions Table (Problem, Decision, Why, Owner) ---
                                                   Problem Decision                                                                                                                                                   Why                                          Owner
                 Pickups outside Jan 2021 (2008, Feb 2021)     DROP    Preserves temporal partition boundary integrity. Downstream January analytics and billing aggregations will be corrupted by out-of-boundary rides. Data Engineering / Ingestion Pipeline Contract
                        Dropoff before pickup (5,642 rows)     DROP                       Negative duration violates physical reality and causality. Distorts transit speed, route modeling, and driver turnaround times.           Data Engineering / Quality Assurance
                               Negative fares (6,769 rows)     DROP          Represents chargebacks, disputes, meter cancellations, or

## Step 5: Clean
Write `clean(df)`. It returns the cleaned dataframe AND a dict `{rule: rows_removed}`.  
Never drop rows silently; do not change the original dataframe.

In [5]:
from clean_taxi_data import clean

chunksize = 100000
total_before = 0
total_after = 0
cum_metrics = {}
parse_dates = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']

for chunk in pd.read_csv('yellow_tripdata_2021-01.csv.gz', chunksize=chunksize, parse_dates=parse_dates, low_memory=False):
    total_before += len(chunk)
    c_clean, c_metrics = clean(chunk)
    total_after += len(c_clean)
    for r, cnt in c_metrics.items():
        cum_metrics[r] = cum_metrics.get(r, 0) + cnt

total_removed = total_before - total_after
pct_removed = (total_removed / total_before) * 100

print('--- PRINTED: CLEANING EXECUTION RESULTS ---')
print(f'Rows Before Cleaning : {total_before:,}')
print(f'Rows After Cleaning  : {total_after:,}')
print(f'Rows Removed         : {total_removed:,} ({pct_removed:.2f}%)\n')

metrics_df = pd.DataFrame([{'Cleaning Rule': k, 'Count': v} for k, v in cum_metrics.items()])
print('--- PRINTED TABLE: {rule: rows_removed} ---')
print(metrics_df.to_string(index=False))

print(f'\nVerification Check: Total Before == Total After + Total Removed? {total_before == total_after + total_removed}')


--- PRINTED: CLEANING EXECUTION RESULTS ---
Rows Before Cleaning : 1,369,765
Rows After Cleaning  : 1,246,349
Rows Removed         : 123,416 (9.01%)

--- PRINTED TABLE: {rule: rows_removed} ---
                          Cleaning Rule  Count
                 rule_1_drop_duplicates      0
   rule_2_drop_pickups_outside_jan_2021     24
      rule_3_drop_dropoff_before_pickup   5642
          rule_4_drop_non_positive_fare   7394
      rule_5_drop_non_positive_distance  18967
      rule_6_drop_missing_critical_keys  91389
rule_7_fix_passenger_count_zero_imputed  26246

Verification Check: Total Before == Total After + Total Removed? True


## Step 6: Stop Here
**Pipeline Stopped**: We do not load the cleaned dataset into PostgreSQL. Data validation and cleaning rules have passed. Loading will take place when connected to the automated Docker pipeline in the next lab session.